In [12]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor


DATA_FOLDER = "./"

df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_light_best_customers_50c.pickle")
#df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_light_grouped.pickle")
if not "serie_id" in df.columns:
    df["serie_id"] = df["product_id"].astype(str) + "_" + df["customer_id"].astype(str)

product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()

In [13]:
import numpy as np
numeric_df = df.select_dtypes(include=[np.number])
total_infs = np.isinf(numeric_df.values).sum()
print(f"Total infs: {total_infs}")
df[numeric_df.columns] = df[numeric_df.columns].replace([np.inf, -np.inf], np.nan)
df['fecha'] = df['fecha'].apply(lambda x: x.to_timestamp('M'))  # último día del mes


Total infs: 0


In [14]:
TEST_DATE = 35

def get_indexes(df, test_date=TEST_DATE):
    test_index = df.index[df['date_id'] == test_date]
    train_index = df.index[df['date_id'] <= test_date]
    train_scaler_index = df.index[df['date_id'] <= test_date]
    return test_index, train_index, train_scaler_index
df["target"] = df.groupby(['customer_id', 'product_id'])['tn'].shift(-2)

test_index, train_index, train_scaler_index = get_indexes(df)
train_df = df.loc[train_index].copy()
test_df = df.loc[test_index].copy()

/tmp/ipykernel_109478/2186123290.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = df.groupby(['customer_id', 'product_id'])['tn'].shift(-2)


In [15]:

static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF

train_df_no_static = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size'])


# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_no_static.columns:
    if train_df_no_static[column].dtype == "int16" or train_df_no_static[column].dtype == "int8":
        train_df_no_static[column] = train_df_no_static[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()

product_id    0
cat1          0
cat2          0
cat3          0
brand         0
sku_size      0
dtype: int64

In [16]:
agg_dict = {}
for col in train_df_no_static.columns:
    if col in ['product_id', 'fecha']:
        continue
    elif col == 'tn':
        agg_dict[col] = 'sum'
    elif train_df_no_static[col].dtype == 'O':
        agg_dict[col] = 'first'
    else:
        agg_dict[col] = 'mean'

# Calcular ticket_promedio y ticket_promedio_con_ceros, total_customers y total_customer_con_ceros
def custom_agg(group):
    tn_no_cero = group[group['tn'] != 0]
    ticket_promedio = tn_no_cero['tn'].mean() if not tn_no_cero.empty else 0
    ticket_promedio_con_ceros = group['tn'].mean()
    total_customers = tn_no_cero['customer_id'].nunique()
    total_customer_con_ceros = group['customer_id'].nunique()
    return pd.Series({
        'ticket_promedio': ticket_promedio,
        'ticket_promedio_con_ceros': ticket_promedio_con_ceros,
        'total_customers': total_customers,
        'total_customer_con_ceros': total_customer_con_ceros
    })

agg_dict_custom = {**agg_dict}
train_df_grouped = train_df_no_static.groupby(['product_id', 'fecha']).agg(agg_dict_custom).reset_index()

# Merge custom aggregations
custom_aggs = train_df_no_static.groupby(['product_id', 'fecha']).apply(custom_agg).reset_index()
train_df_grouped = train_df_grouped.merge(custom_aggs, on=['product_id', 'fecha'], how='left')
train_df_grouped

/tmp/ipykernel_109478/3666341461.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  custom_aggs = train_df_no_static.groupby(['product_id', 'fecha']).apply(custom_agg).reset_index()


,product_id,fecha,customer_id,periodo_min_producto,periodo_max_producto,periodo_min_customer,periodo_max_customer,plan_precios_cuidados,cust_request_qty,cust_request_tn,...,tn_total_vendidas,tn_customer_weight,tn_product_vendidas,tn_product_weight,serie_id,target,ticket_promedio,ticket_promedio_con_ceros,total_customers,total_customer_con_ceros
0,20001,2017-01-31,9828.960784,2017-01-01,2019-12-01,2017-01-01,2019-11-27 08:09:36,0.0,9.392157,18.386806,...,34057.316406,0.019608,934.772217,2.744703e-02,20001_0,25.556032,23.968519,18.328867,39.0,51.0
1,20001,2017-02-28,9828.960784,2017-01-01,2019-12-01,2017-01-01,2019-11-27 08:09:36,0.0,8.470588,16.347488,...,34568.652344,0.019608,798.016174,2.308496e-02,20001_0,20.979633,19.000385,15.647376,42.0,51.0
2,20001,2017-03-31,9828.960784,2017-01-01,2019-12-01,2017-01-01,2019-11-27 08:09:36,0.0,9.980392,26.093077,...,46040.597656,0.019608,1303.357666,2.830888e-02,20001_0,29.454927,34.298885,25.556036,38.0,51.0
3,20001,2017-04-30,9828.960784,2017-01-01,2019-12-01,2017-01-01,2019-11-27 08:09:36,0.0,5.470588,22.214594,...,39625.523438,0.019608,1069.961304,2.700182e-02,20001_0,29.805204,32.423069,20.979633,33.0,51.0
4,20001,2017-05-31,9828.960784,2017-01-01,2019-12-01,2017-01-01,2019-11-27 08:09:36,0.0,13.745098,30.405674,...,45579.632812,0.019608,1502.201294,3.295773e-02,20001_0,20.209291,34.140938,29.454927,44.0,51.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31517,21295,2017-01-31,9828.960784,2017-01-01,2017-01-01,2017-01-01,2019-11-27 08:09:36,0.0,0.019608,0.000137,...,34057.316406,0.019608,0.006990,2.052422e-07,21295_0,NaN,0.006990,0.000137,1.0,51.0
31518,21296,2017-08-31,9828.960784,2017-08-01,2017-08-01,2017-01-01,2019-11-27 08:09:36,0.0,0.019608,0.000128,...,40206.382812,0.019608,0.006510,1.619146e-07,21296_0,NaN,0.006510,0.000128,1.0,51.0
31519,21297,2017-01-31,9828.960784,2017-01-01,2017-01-01,2017-01-01,2019-11-27 08:09:36,0.0,0.019608,0.000114,...,34057.316406,0.019608,0.005790,1.700075e-07,21297_0,NaN,0.005790,0.000114,1.0,51.0
31520,21298,2017-08-31,9828.960784,2017-08-01,2017-08-01,2017-01-01,2019-11-27 08:09:36,0.0,0.019608,0.000112,...,40206.382812,0.019608,0.005730,1.425147e-07,21298_0,NaN,0.005730,0.000112,1.0,51.0


In [17]:

static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF


# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_no_static.columns:
    if train_df_no_static[column].dtype == "int16" or train_df_no_static[column].dtype == "int8":
        train_df_no_static[column] = train_df_no_static[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()


train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_grouped.drop(columns=['target'], errors="ignore"),  # No necesito la columna target en train_data
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)
train_data.head()


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="ticket_promedio",
    freq="ME",
)


predictor.fit(
    train_data,
    #presets="best_quality",
    presets="medium_quality",
    time_limit=600,
)
preds_tn_promedio = predictor.predict(train_data)


Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250701_232455'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       4.18 GB / 15.32 GB (27.3%)
Disk Space Avail:   67.85 GB / 575.67 GB (11.8%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'ticket_promedio',
 'time_limit': 600,
 'verbosity': 2}

train_

In [18]:

static_features_df = pd.DataFrame({
    'cat1': df.groupby('product_id')['cat1'].first(),
    'cat2': df.groupby('product_id')['cat2'].first(),
    'cat3': df.groupby('product_id')['cat3'].first(),
    'brand': df.groupby('product_id')['brand'].first(),
    'sku_size': df.groupby('product_id')['sku_size'].first(),
}).reset_index()
# drop this from test)DF
 
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()

train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_grouped.drop(columns=['target'], errors="ignore"),  # No necesito la columna target en train_data
    id_column="product_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)
train_data.head()


predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="total_customers",
    freq="ME",
)


predictor.fit(
    train_data,
    presets="medium_quality",
    time_limit=600,
)
preds_total_customer = predictor.predict(train_data)

Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250701_233043'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       2.66 GB / 15.32 GB (17.4%)
Disk Space Avail:   67.70 GB / 575.67 GB (11.8%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'ME',
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'total_customers',
 'time_limit': 600,
 'verbosity': 2}

train_

In [22]:
# la prediccion es la columna mean del ultimo timestamp para cada product_id
predictions = preds_tn_promedio.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'product_id', "mean": "mean_ticket_promedio"})
predictions["total_customers"] = preds_total_customer.groupby('item_id').last()['mean'].reset_index()['mean']

predictions["prediction"] = predictions["mean_ticket_promedio"] * (predictions["total_customers"]-0)
# le agrego la columna target de test_df mergeando por product_id
test_df_grouped = test_df.groupby(['product_id', 'fecha']).agg({
    'target': 'sum'
}).reset_index()
predictions = predictions.merge(test_df_grouped[['product_id', 'target']], on='product_id', how='left')
predictions = predictions[predictions["product_id"].isin(product_ids)]



predictions["abs_error"] = abs(predictions['target'] - predictions['prediction'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['prediction']).sum() / (predictions['target'].sum())
print(f"Total Absolute Error: {total_error:.4f}")
predictions

Total Absolute Error: inf


/tmp/ipykernel_109478/4095548758.py:18: RuntimeWarning: divide by zero encountered in scalar divide
  total_error = np.abs(predictions['target'] - predictions['prediction']).sum() / (predictions['target'].sum())


,product_id,mean_ticket_promedio,total_customers,prediction,target,abs_error
0,20001,35.284177,37.570683,1325.650625,0.0,1325.650625
1,20002,30.379225,34.485828,1047.652718,0.0,1047.652718
2,20003,20.930870,36.679711,767.738278,0.0,767.738278
3,20004,15.183389,36.030392,547.063458,0.0,547.063458
4,20005,17.220930,33.255283,572.686912,0.0,572.686912
...,...,...,...,...,...,...
1197,21263,-0.004804,3.177319,-0.015265,0.0,0.015265
1199,21265,0.019684,3.066418,0.060358,0.0,0.060358
1200,21266,0.017417,3.002886,0.052300,0.0,0.052300
1201,21267,0.019250,1.999940,0.038498,0.0,0.038498


## en test: Total Absolute Error: 0.2178

In [24]:
submission = predictions[['product_id', 'prediction']]
submission.rename(columns={'prediction': 'tn'}, inplace=True)
submission.to_csv(DATA_FOLDER + "submission_autogluon_mean_ticket0.csv", index=False)

/tmp/ipykernel_109478/1844313901.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  submission.rename(columns={'prediction': 'tn'}, inplace=True)
